<a href="https://colab.research.google.com/github/sarangcsoman/dataviz-exercises-sarang/blob/main/quiz/dataviz_quiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [9]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I chose a line chart because it clearly shows population growth over time for each continent. I used 1970 as the baseline (100%) to compare relative population growth fairly. Africa shows the fastest growth.

In [10]:
# Q1

continent_pop = df_long.groupby(
    ['Continent', 'Year']
)['Population'].sum().reset_index()

baseline = continent_pop[
    continent_pop['Year'] == 1970
][['Continent', 'Population']].rename(
    columns={'Population': 'Population_1970'}
)

continent_pop = continent_pop.merge(baseline, on='Continent')

continent_pop['Growth_Index'] = (
    continent_pop['Population'] /
    continent_pop['Population_1970'] * 100
)

fig = px.line(
    continent_pop,
    x='Year',
    y='Growth_Index',
    color='Continent',
    title='Africa Has Experienced the Fastest Population Growth Since 1970',
    markers=True,
    template='plotly_white'
)

fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Population Growth Index (1970 = 100)',
    legend_title='Continent'
)

fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I chose a horizontal sorted bar chart because it clearly compares the 15 countries with the largest population increases between 1970 and 2022. I used different colours to highlight African countries. Four of the 15 countries are in Africa.

In [11]:
# Q2
# YOUR CODE HERE
# Q2

df['Population Added'] = (
    df['2022 Population'] - df['1970 Population']
)

top15 = df.nlargest(15, 'Population Added').sort_values(
    'Population Added', ascending=True
)

top15['Region'] = top15['Continent'].apply(
    lambda x: 'Africa' if x == 'Africa' else 'Other continents'
)

fig = px.bar(
    top15,
    x='Population Added',
    y='Country',
    orientation='h',
    color='Region',
    color_discrete_map={
        'Africa': '#0072B2',
        'Other continents': '#999999'
    },
    title='Four African Countries Are Among the Top 15 Population Contributors',
    template='plotly_white'
)

fig.update_layout(
    xaxis_title='Population Added (1970–2022)',
    yaxis_title='Country',
    legend_title='Region'
)

fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I chose a scatter plot because it shows the relationship between land area and population across countries. I used different colours for each continent to compare regional patterns. Logarithmic scales make differences easier to see, while country labels highlight outliers such as India, Bangladesh, Russia, Canada and Australia.

In [12]:
# Q3
# YOUR CODE HERE
# Q3

outliers = ['India', 'Bangladesh', 'Russia', 'Canada', 'Australia']

df['Label'] = df['Country'].where(
    df['Country'].isin(outliers), ''
)

fig = px.scatter(
    df,
    x='Area',
    y='2022 Population',
    color='Continent',
    hover_name='Country',
    text='Label',
    log_x=True,
    log_y=True,
    color_discrete_map={
        'Africa': '#0072B2',
        'Asia': '#E69F00',
        'Europe': '#999999',
        'North America': '#56B4E9',
        'Oceania': '#CC79A7',
        'South America': '#332288'
    },
    title='Land Area and Population Show Different Patterns Across Continents',
    template='plotly_white'
)

fig.update_traces(
    textposition='top center',
    marker=dict(size=9, opacity=0.75)
)

fig.update_layout(
    xaxis_title='Land Area (km²) – Log Scale',
    yaxis_title='2022 Population – Log Scale',
    legend_title='Continent'
)

fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I chose a violin plot because it shows the distribution and variation of population density across continents. It helps compare how densely populated countries are in different continents and highlights differences in their distributions.

In [13]:
# Q4
# YOUR CODE HERE
# Q4

df['Population Density'] = (
    df['2022 Population'] / df['Area']
)

fig = px.violin(
    df,
    x='Continent',
    y='Population Density',
    color='Continent',
    box=True,
    points='outliers',
    log_y=True,
    title='Population Density Distribution Across Continents',
    template='plotly_white'
)

fig.update_layout(
    xaxis_title='Continent',
    yaxis_title='Population Density (People per km², Log Scale)',
    showlegend=False
)

fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I chose a choropleth map because it displays population growth geographically across the world. Different colours represent population changes, making it easy to identify countries with increasing or decreasing populations.

In [14]:
# Q5
# YOUR CODE HERE
# Q5

df['Percentage Change'] = (
    (df['2022 Population'] - df['2000 Population'])
    / df['2000 Population'] * 100
)

fig = px.choropleth(
    df,
    locations='CCA3',
    color='Percentage Change',
    hover_name='Country',
    color_continuous_scale='RdYlGn',
    title='Population Percentage Change by Country (2000–2022)',
    template='plotly_white'
)

fig.update_layout(
    geo=dict(showframe=False, showcoastlines=True),
    coloraxis_colorbar_title='Change (%)'
)

fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I chose a treemap because it shows the distribution of the 2022 world population at two levels: continents and individual countries. The size of each rectangle represents population, making it easy to compare continents and identify the countries with the largest populations.

In [15]:
# Q6
# YOUR CODE HERE
# Q6

fig = px.treemap(
    df,
    path=['Continent', 'Country'],
    values='2022 Population',
    title='Distribution of the 2022 World Population by Continent and Country',
    color='2022 Population',
    color_continuous_scale='Blues',
    template='plotly_white'
)

fig.update_traces(
    textinfo='label+percent parent'
)

fig.update_layout(
    margin=dict(t=50, l=10, r=10, b=10)
)

fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I chose a 100% stacked bar chart because it compares the percentage share of world population across continents in 1970 and 2022. Africa, Asia and Europe are shown separately, while the remaining continents are grouped as 'Rest of the world'. This makes changes in population share easy to compare.

In [17]:
# Q7
# YOUR CODE HERE
# Q7

q7_data = df_long[
    df_long['Year'].isin([1970, 2022])
].copy()

q7_data['Group'] = q7_data['Continent'].apply(
    lambda x: x if x in ['Africa', 'Asia', 'Europe']
    else 'Rest of the world'
)

q7_grouped = q7_data.groupby(
    ['Year', 'Group']
)['Population'].sum().reset_index()

q7_grouped['Share'] = (
    q7_grouped['Population'] /
    q7_grouped.groupby('Year')['Population'].transform('sum')
) * 100
q7_grouped['Year'] = q7_grouped['Year'].astype(str)

fig = px.bar(
    q7_grouped,
    x='Year',
    y='Share',
    color='Group',
    barmode='stack',
    title='How World Population Shares Changed Between 1970 and 2022',
    color_discrete_map={
        'Africa': '#009E73',
        'Asia': '#E69F00',
        'Europe': '#0072B2',
        'Rest of the world': '#999999'
    },
    template='plotly_white'
)

fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Share of World Population (%)',
    yaxis=dict(range=[0, 100]),
    legend_title='Continent'
)

fig.show()